# Multi-Modal Physics-Informed SwinIR for Indian Ocean Magnetics Enhancement
### Full-Fledged Training Pipeline on Kaggle GPU (T4 / P100 / A100)
This notebook trains an official **SwinIR (Swin Transformer)** model on the Indian Ocean basin combining:
1. **EMAG2 v3 Marine Magnetic Anomaly Field**
2. **ETOPO 2022 Seafloor Bathymetry**
3. **SWOT Satellite Free-Air Gravity Anomaly**
4. **Gallardo & Meju (2003) Structural Cross-Gradient Physics Loss**

In [ ]:
# 1. Install required dependencies
!pip install -q timm rasterio xarray netcdf4 scikit-image

In [ ]:
# 2. Verify GPU acceleration
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    device = torch.device('cuda')
else:
    print("WARNING: Running on CPU. Please enable GPU accelerator in Kaggle Notebook Settings!")
    device = torch.device('cpu')

In [ ]:
# 3. Clone / Import Official SwinIR Architecture
import os, sys
if not os.path.exists('SwinIR'):
    !git clone https://github.com/JingyunLiang/SwinIR.git
sys.path.append('SwinIR')
from models.network_swinir import SwinIR

In [ ]:
# 4. Physics-Informed Loss Definition
import torch.nn as nn
import torch.nn.functional as F

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super(CharbonnierLoss, self).__init__()
        self.eps = eps
    def forward(self, x, y):
        diff = x - y
        return torch.mean(torch.sqrt(diff * diff + self.eps))

class CrossGradientLoss(nn.Module):
    def __init__(self):
        super(CrossGradientLoss, self).__init__()
        kx = torch.tensor([[-0.5, 0.0, 0.5]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        ky = torch.tensor([[-0.5], [0.0], [0.5]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        self.register_buffer('kernel_x', kx)
        self.register_buffer('kernel_y', ky)
    def forward(self, m, z):
        dm_dx = F.conv2d(F.pad(m, (1, 1, 0, 0), mode='replicate'), self.kernel_x)
        dm_dy = F.conv2d(F.pad(m, (0, 0, 1, 1), mode='replicate'), self.kernel_y)
        dz_dx = F.conv2d(F.pad(z, (1, 1, 0, 0), mode='replicate'), self.kernel_x)
        dz_dy = F.conv2d(F.pad(z, (0, 0, 1, 1), mode='replicate'), self.kernel_y)
        cross_grad = dm_dx * dz_dy - dm_dy * dz_dx
        return torch.mean(torch.abs(cross_grad))

class GeophysicalPhysicsLoss(nn.Module):
    def __init__(self, lambda_cg=0.08, lambda_tv=0.002):
        super(GeophysicalPhysicsLoss, self).__init__()
        self.recon = CharbonnierLoss()
        self.cg = CrossGradientLoss()
        self.lambda_cg = lambda_cg
        self.lambda_tv = lambda_tv
    def forward(self, pred, target, guide):
        l_rec = self.recon(pred, target)
        l_cg = self.cg(pred, guide)
        diff_h = torch.abs(pred[:, :, 1:, :] - pred[:, :, :-1, :])
        diff_w = torch.abs(pred[:, :, :, 1:] - pred[:, :, :, :-1])
        l_tv = torch.mean(diff_h) + torch.mean(diff_w)
        return l_rec + self.lambda_cg * l_cg + self.lambda_tv * l_tv, l_rec, l_cg

In [ ]:
# 5. Full-Fledged Dataset Loader with Dynamic Indian Ocean Sampling
import rasterio, rasterio.windows, xarray as xr, numpy as np
from torch.utils.data import Dataset, DataLoader

class KaggleIndianOceanDataset(Dataset):
    def __init__(self, emag_path, bathy_path, grav_path, patch_size=64, num_samples=2000, seed=42):
        super().__init__()
        self.patch_size = patch_size
        self.num_samples = num_samples
        np.random.seed(seed)
        self.src_emag = rasterio.open(emag_path)
        self.ds_bathy = xr.open_dataset(bathy_path)
        self.ds_grav = xr.open_dataset(grav_path)
        self.lon_min, self.lon_max = 25.0, 140.0
        self.lat_min, self.lat_max = -55.0, 25.0
        self.coords = []
        print(f"Generating {num_samples} valid deep-ocean training patches...")
        while len(self.coords) < num_samples:
            lon_c = np.random.uniform(self.lon_min, self.lon_max)
            lat_c = np.random.uniform(self.lat_min, self.lat_max)
            d_deg = patch_size * 0.0167
            b1, b2, b3, b4 = lon_c - d_deg/2, lat_c - d_deg/2, lon_c + d_deg/2, lat_c + d_deg/2
            try:
                sub_z = self.ds_bathy.z.sel(lat=slice(b2, b4), lon=slice(b1, b3)).values
                if sub_z.size > 0 and np.nanmean(sub_z) < -200:
                    self.coords.append((b1, b2, b3, b4))
            except:
                continue
        print(f"Ready! Total training samples: {len(self.coords)}")

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        b1, b2, b3, b4 = self.coords[idx]
        win = rasterio.windows.from_bounds(b1, b2, b3, b4, self.src_emag.transform)
        m_hr = self.src_emag.read(1, window=win, out_shape=(self.patch_size, self.patch_size), resampling=rasterio.enums.Resampling.bilinear).astype(np.float32)
        m_lr = self.src_emag.read(1, window=win, out_shape=(self.patch_size//2, self.patch_size//2), resampling=rasterio.enums.Resampling.cubic).astype(np.float32)
        sub_b = self.ds_bathy.z.sel(lat=slice(b2, b4), lon=slice(b1, b3)).values
        sub_g = self.ds_grav.z.sel(lat=slice(b2, b4), lon=slice(b1, b3)).values
        b_hr = F.interpolate(torch.tensor(sub_b, dtype=torch.float32).unsqueeze(0).unsqueeze(0), size=(self.patch_size, self.patch_size), mode='bilinear').squeeze().numpy()
        g_hr = F.interpolate(torch.tensor(sub_g, dtype=torch.float32).unsqueeze(0).unsqueeze(0), size=(self.patch_size, self.patch_size), mode='bilinear').squeeze().numpy()
        def norm(a):
            a = np.nan_to_num(a)
            mi, ma = np.min(a), np.max(a)
            return (a - mi) / (ma - mi + 1e-7)
        m_up = F.interpolate(torch.tensor(norm(m_lr)).unsqueeze(0).unsqueeze(0), size=(self.patch_size, self.patch_size), mode='bicubic').squeeze().numpy()
        x_in = np.stack([m_up, norm(b_hr), norm(g_hr)], axis=0).astype(np.float32)
        y_tgt = np.expand_dims(norm(m_hr), axis=0).astype(np.float32)
        b_guide = np.expand_dims(norm(b_hr), axis=0).astype(np.float32)
        return torch.tensor(x_in), torch.tensor(y_tgt), torch.tensor(b_guide)

In [ ]:
# 6. Full Training Loop (50 - 100 Epochs on GPU)
# Update dataset paths to wherever you extract the zip in Kaggle (/kaggle/input/... or current dir)
data_dir = '.'
emag_p = os.path.join(data_dir, 'EMAG2_V3_SeaLevel_DataTiff.tif')
bathy_p = os.path.join(data_dir, 'ETOPO_2022_v1_60s_N90W180_bed.nc')
grav_p = os.path.join(data_dir, 'grav_SWOT_05.nc')

patch_size = 64
batch_size = 16  # Higher batch size on GPU
num_samples = 2000  # Full-fledged dataset
epochs = 50
lr = 3e-4

train_ds = KaggleIndianOceanDataset(emag_p, bathy_p, grav_p, patch_size=patch_size, num_samples=num_samples)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)

model = SwinIR(
    upscale=1, in_chans=3, img_size=patch_size, window_size=8, img_range=1.0,
    depths=[6, 6, 6, 6], embed_dim=60, num_heads=[6, 6, 6, 6], mlp_ratio=2, upsampler=''
).to(device)
head = nn.Conv2d(3, 1, kernel_size=3, padding=1).to(device)

criterion = GeophysicalPhysicsLoss(lambda_cg=0.08, lambda_tv=0.002).to(device)
optimizer = torch.optim.AdamW(list(model.parameters()) + list(head.parameters()), lr=lr, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

print("Starting Full-Fledged Training on GPU...")
for epoch in range(1, epochs + 1):
    model.train()
    head.train()
    total_l, steps = 0.0, 0
    for x, y, b in train_loader:
        x, y, b = x.to(device), y.to(device), b.to(device)
        optimizer.zero_grad()
        pred = head(model(x))
        loss, r_l, cg_l = criterion(pred, y, b)
        loss.backward()
        optimizer.step()
        total_l += loss.item()
        steps += 1
    scheduler.step()
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{epochs}] | Loss: {total_l/steps:.5f} | LR: {scheduler.get_last_lr()[0]:.6f}")

torch.save({'model': model.state_dict(), 'head': head.state_dict()}, 'swinir_indian_ocean_kaggle_final.pth')
print("Training Complete! Saved swinir_indian_ocean_kaggle_final.pth")